In [0]:
%sql
-- Creating a catalog and schema 

Create catalog if not exists cdc_catalog;
create schema if not exists cdc_catalog.cdc_schema; 
use  cdc_catalog.cdc_schema;



In [0]:
#Checking the path 

print('The below path will drop the checkout path...by default it will be disabled')
chkpnt_path = '/Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint'
print(f"The checkpoint path :-------- {chkpnt_path}")
#dbutils.fs.rm(f"{chkpnt_path}",recurse=True)

The below path will drop the checkout path...by default it will be disabled
The checkpoint path :-------- /Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint


In [0]:
%sql
/*
DROP TABLE customer_base_table;
*/

Create or replace table customer_base_table(
  userId  INT,name STRING , city STRING)
  USING DELTA 
  TBLPROPERTIES (delta.enableChangeDataFeed = true,
                 delta.deletedFileRetentionDuration = 'interval 2 days');

/* incase customer_base_table already created
ALTER TABLE customer_base_table 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
*/


In [0]:
%sql
DESCRIBE TABLE EXTENDED customer_base_table


col_name,data_type,comment
userId,int,null
name,string,null
city,string,null
,,
# Detailed Table Information,,
Catalog,cdc_catalog,
Database,cdc_schema,
Table,customer_base_table,
Created Time,Fri Jul 10 06:35:46 UTC 2026,
Last Access,UNKNOWN,


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_base_table", 14)

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,delete,14,2026-07-13T10:47:41.000Z


In [0]:
%sql
Insert into customer_base_table
VALUES (101, "Raul", "Oaxaca")

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_base_table", 0)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8395774189779189>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', ' SELECT * \n   FROM table_changes("customer_base_table", 0)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.d

In [0]:
%sql
VACUUM customer_base_table;

path
""


In [0]:
%sql
DESCRIBE HISTORY customer_base_table


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
13,2026-07-13T10:45:09.000Z,7912518962361895,sandutta2020@gmail.com,VACUUM END,Map(status -> COMPLETED),null,List(534018730120359),884843ff-8045-4c90-8fab-c33582f83c5b,0713-103249-u2h24wok-v2n,12,SnapshotIsolation,true,"Map(numDeletedFiles -> 6, numVacuumedDirectories -> 2)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
12,2026-07-13T10:45:08.000Z,7912518962361895,sandutta2020@gmail.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 86400000)",null,List(534018730120359),884843ff-8045-4c90-8fab-c33582f83c5b,0713-103249-u2h24wok-v2n,11,SnapshotIsolation,true,"Map(numFilesToDelete -> 6, sizeOfDataToDelete -> 7063)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
11,2026-07-13T10:40:54.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),5aae284a-d3c2-4d6c-8471-394a4077d76a,0713-103249-u2h24wok-v2n,10,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
10,2026-07-13T10:33:53.000Z,7912518962361895,sandutta2020@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.deletedFileRetentionDuration"":""interval 1 days"",""delta.parquet.format.version.afe.internal"":""2.12.0""}, statsOnLoad -> false)",null,List(534018730120359),ac8a0ec3-4808-4244-8a3c-adf231c8e848,0713-103249-u2h24wok-v2n,9,WriteSerializable,false,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
9,2026-07-13T08:50:34.000Z,7912518962361895,sandutta2020@gmail.com,TRUNCATE,Map(),null,List(534018730120359),a533fb35-d443-4f3e-82a7-c0ee6d53f065,0713-084908-h9ecvwb8-v2n,8,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1327, numDeletionVectorsRemoved -> 0, executionTimeMs -> 219, numDeletedRows -> 4)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
8,2026-07-13T07:59:54.000Z,7912518962361895,sandutta2020@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(534018730120359),f06821bc-6015-4e04-9c25-9a26556a2c3c,0713-075022-s91oy0zo-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2551, p25FileSize -> 1327, numDeletionVectorsRemoved -> 1, minFileSize -> 1327, numAddedFiles -> 1, maxFileSize -> 1327, p75FileSize -> 1327, p50FileSize -> 1327, numAddedBytes -> 1327)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-07-13T07:59:50.000Z,7912518962361895,sandutta2020@gmail.com,DELETE,"Map(predicate -> [""(userId#11665 = 102)""])",null,List(534018730120359),f06821bc-6015-4e04-9c25-9a26556a2c3c,0713-075022-s91oy0zo-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3788, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 2737, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 1000)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-13T07:59:44.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),e1a719da-9f50-481a-a636-5aa98dc373e9,0713-075022-s91oy0zo-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1236)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-10T06:42:39.000Z,7912518962361895,sandutta2020@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrd

In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_base_table", 1)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4820965654941768>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', ' SELECT * \n   FROM table_changes("customer_base_table", 1)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.d

In [0]:
%sql
insert into customer_base_table VALUES
(102, "Jhon", "Mexico")

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_table", 1)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4820965654941770>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', ' SELECT * \n   FROM table_changes("customer_table", 1)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver

In [0]:
%sql
DESCRIBE HISTORY customer_table


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-07-09T11:58:20.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),bfa75715-9ae2-4904-960e-96ea7addf935,0709-115413-tcv7z9j4-v2n,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1240)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
6,2026-07-09T11:57:05.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),473da267-e4de-4710-a42e-e3ea068325c9,0709-115413-tcv7z9j4-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1219)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
5,2026-07-09T11:55:52.000Z,7912518962361895,sandutta2020@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(534018730120359),b54fb85c-96c2-4c8f-a3fe-aa4b9ba7580d,0709-115413-tcv7z9j4-v2n,4,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
4,2026-07-09T11:55:40.000Z,7912518962361895,sandutta2020@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0""}, statsOnLoad -> true)",null,List(534018730120359),b7b93227-d3ff-4edf-9072-90aabe72740b,0709-115413-tcv7z9j4-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 3, numRemovedBytes -> 3678, numDeletionVectorsRemoved -> 0, numOutputRows -> 1, numOutputBytes -> 1219)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
3,2026-07-09T11:25:43.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),0c2e6b5d-b1e8-409d-96af-7a3dd47a238b,0709-111222-8uzg0lye-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1240)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-07-09T11:22:29.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),7cd0f516-8af5-4b96-9c04-23a286f8f501,0709-111222-8uzg0lye-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1219)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-07-09T11:17:46.000Z,7912518962361895,sandutta2020@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(534018730120359),1b44d7e6-16fc-4447-bde9-316b609021c4,0709-111222-8uzg0lye-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-07-09T11:17:09.000Z,7912518962361895,sandutta2020@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(534018730120359),2ceef416-2e91-4825-b6d4-6b513ad26167,0709-111222-8uzg0lye-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1219)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
%sql
insert into customer_base_table VALUES
(103, "Lily", "Canada"),
(104, "colin", "USA")

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
select * from customer_base_table

userId,name,city
101,Raul,Oaxaca
104,colin,USA
103,Lily,New York
105,mike,Colombo


In [0]:
%sql


insert into customer_base_table values(105, "mike", "Colombo") ;
delete from customer_base_table where userId = 102;




num_affected_rows
1


In [0]:
%sql
Truncate table cdc_catalog.cdc_schema.customer_base_table

In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_table", 5)

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,insert,7,2026-07-09T11:58:20.000Z
104,colin,USA,insert,7,2026-07-09T11:58:20.000Z
102,Jhon,Mexico,insert,6,2026-07-09T11:57:05.000Z


In [0]:
cdf_df = (spark.readStream
               .format("delta")
               .option("readChangeData", True)
               .option("startingVersion", 5)
               .table("customer_table"))

display(cdf_df, checkpointLocation = "/Volumes/dbx_catalog/dbx_schema/cdf_checkpoint")




Checkpointing to /Volumes/dbx_catalog/dbx_schema/cdf_checkpoint


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5825276710950743>, line 7
      1 cdf_df = (spark.readStream
      2                .format("delta")
      3                .option("readChangeData", True)
      4                .option("startingVersion", 5)
      5                .table("customer_table"))
----> 7 display(cdf_df, checkpointLocation = "/Volumes/dbx_catalog/dbx_schema/cdf_checkpoint")

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     92     raise t

In [0]:
dbutils.fs.rm("/Volumes/dbx_catalog/dbx_schema/cdf_checkpoint", recurse=True)

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-7381045359658502>, line 1
----> 1 dbutils.fs.rm("/Volumes/dbx_catalog/dbx_schema/cdf_checkpoint", recurse=True)

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     59 ) from None

ExecutionError: [UC_VOLUME_NOT_FOUND] Volume `dbx_catalog`.`dbx_schema`.`cdf_checkpoint` does not exist. Please use 'SHOW VOLUMES' to list available volumes. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.NoSuchVolumeException
	at com.databricks.sql.managedcatalog.client.M

In [0]:
spark.readStream.option("readChangeFeed", "true").option("startingVersion", 1).table("customer_base_table").writeStream.option("checkpointLocation", "/Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint").trigger (availableNow=True).table("customers_updates")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7381045359658501>, line 1
----> 1 spark.readStream.option("readChangeFeed", "true").option("startingVersion", 1).table("customer_base_table").writeStream.option("checkpointLocation", "/Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint").trigger (availableNow=True).table("customers_updates")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:737, in DataStreamWriter.table(self, tableName)
    735 def table(self, tableName: str) -> "StreamingQuery":
    736     """Alias for the toTable API"""
--> 737     return self.toTable(tableName)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:748, in DataStreamWriter.toTable(self, tableName, format, outputMode, partitionBy, queryName, **options)
    739 def toTable(
    740     sel

In [0]:
%sql
select * from cdc_catalog.information_schema.table_privileges

grantor,grantee,table_catalog,table_schema,table_name,privilege_type,is_grantable,inherited_from
System user,account users,cdc_catalog,information_schema,schema_tags,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,volumes,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,schema_privileges,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,table_tags,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,referential_constraints,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,volume_privileges,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,constraint_table_usage,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,views,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,catalog_properties,SELECT,NO,NONE
System user,account users,cdc_catalog,information_schema,columns,SELECT,NO,NONE


In [0]:
%sql
select * from customers_updates

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,update_preimage,4,2026-07-10T06:42:36.000Z
103,Lily,New York,update_postimage,4,2026-07-10T06:42:36.000Z
103,Lily,Canada,insert,3,2026-07-10T06:39:54.000Z
104,colin,USA,insert,3,2026-07-10T06:39:54.000Z
105,mike,Colombo,insert,6,2026-07-13T07:59:44.000Z
102,Jhon,Mexico,delete,7,2026-07-13T07:59:50.000Z
102,Jhon,Mexico,insert,2,2026-07-10T06:37:20.000Z


In [0]:
%sql
update customer_base_table set city='New York' where userId=103

num_affected_rows
1


In [0]:
%sql
select * from customers_updates

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,insert,3,2026-07-10T06:39:54.000Z
104,colin,USA,insert,3,2026-07-10T06:39:54.000Z
102,Jhon,Mexico,insert,2,2026-07-10T06:37:20.000Z


In [0]:
spark.readStream.option("readChangeFeed", "true").option("startingVersion", 1).table("customer_base_table").writeStream.option("checkpointLocation", "/Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint").trigger (availableNow=True).table("customers_updates")

In [0]:
%sql
select * from customers_updates

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,delete,9,2026-07-13T08:50:34.000Z
104,colin,USA,delete,9,2026-07-13T08:50:34.000Z
103,Lily,New York,delete,9,2026-07-13T08:50:34.000Z
105,mike,Colombo,delete,9,2026-07-13T08:50:34.000Z
103,Lily,Canada,update_preimage,4,2026-07-10T06:42:36.000Z
103,Lily,New York,update_postimage,4,2026-07-10T06:42:36.000Z
103,Lily,Canada,insert,3,2026-07-10T06:39:54.000Z
104,colin,USA,insert,3,2026-07-10T06:39:54.000Z
105,mike,Colombo,insert,6,2026-07-13T07:59:44.000Z
102,Jhon,Mexico,delete,7,2026-07-13T07:59:50.000Z
